# Co-folding at scale: Boltz-2 and OpenBind-0 on Ray

**AI Drug Discovery & Development Summit 2026 · Hands-on workshop · 90 minutes**

Co-folding models such as Boltz-2 and OpenFold3 predict how a drug candidate sits in its protein target,
its *pose*, from the protein sequence and the molecule's structure. In this workshop you measure how
accurate those predictions are on 3 real antiviral drug discovery campaigns with 2,147 X-ray crystal
structures, and you build the full workflow on Ray: distributed inference, scoring, fine-tuning,
hyperparameter tuning and a live prediction service.

![The workshop end to end](docs/images/workshop.gif)

| Section | Topic | Ray library | Minutes |
|---|---|---|---|
| 0 | Setup | | 3 |
| 1 | Ray fundamentals | Ray Core | 15 |
| 2 | The data: 3 drug discovery campaigns | | 6 |
| 3 | Predict and score on a GPU cluster | Ray Data | 18 |
| 4 | Results across 2,147 crystal structures | | 10 |
| 5 | Fine-tune OpenFold3 for 1 target protein | Ray Train | 13 |
| 6 | Hyperparameter and inference sweeps | Ray Tune | 5 |
| 7 | Live prediction service and 3D viewer | Ray Serve | 15 |
| 8 | Summary and next steps | | 5 |

**Running the notebook.** Run the cells in order with **Shift+Enter**. Cells marked ⏱ take a few minutes
while GPU nodes start and models load; you can keep reading while they run. Scientific terms are explained
in the [glossary in the README](README.md#jargon-in-plain-words).

## 0. Setup

This Workspace uses the `aiddd-boltz2-of3` container image, which includes Ray, Boltz-2 and OpenFold3. The
notebook runs on the head node, a CPU-only machine. NVIDIA L4 GPU nodes join the cluster when a cell
requests a GPU and are released when they are idle. The input data, model weights and the results of the
full benchmark are on shared storage.

In [1]:
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (SHARED, TARGETS, BOLTZ_CACHE, OF3_OB_CKPT, Boltz2Predictor, OF3Predictor,
                            load_ligands, validate, make_scorer)
import workshop_helpers as wh

# Data and weights are downloaded once, ahead of the workshop (README step 2)
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS] + [BOLTZ_CACHE, OF3_OB_CKPT]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"Missing {missing}. Run `python setup_data.py` and `python prepare_mpro.py` first."

# GPU type for every GPU step in this notebook. L4 by default; change it here if the session uses another type.
GPU = "L4"

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")

# Connect to the Workspace's Ray cluster. Every worker gets the same shared-storage path.
try:
    ray.init(runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}}, ignore_reinit_error=True, logging_level="ERROR")
except ValueError:   # running inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = os.environ.get("RAY_OVERRIDE_DASHBOARD_URL") or ray.get_runtime_context().dashboard_url
print(f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)")

shared data:  /mnt/user_storage/aiddd
your outputs: /mnt/user_storage/aiddd/workshop/alicia_chua_30e1ffa
Ray Dashboard metrics: https://session-thqdxs9cduqis7hsp7r4mvd42v.i.anyscaleuserdata.com/#/metrics   (open in a new browser tab)


## 1. Ray fundamentals

A single Boltz-2 prediction takes about 1 minute on 1 GPU. The full benchmark covers 2,147 crystal structures,
3 models and 3 proteins, which adds up to several days on a single GPU. A drug discovery team also needs
to fine-tune models, compare settings and give chemists access to the results.

Ray is an open-source framework for running Python across a cluster of machines. Ray Core provides a small
set of primitives (tasks, actors and objects). The Ray AI libraries for data processing, training, tuning
and serving are built on top of them. This workshop uses all 4 libraries.

<img src="docs/images/ray/map-of-ray.svg" alt="Ray overview" width="460">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

### 1.1 The cluster

A Ray cluster has 1 **head node** and any number of **worker nodes**. Your notebook is the *driver*
process on the head node. Each node runs a scheduler and an **object store**, a shared-memory area that
holds results so other processes can read them without copying. The **autoscaler** on the head node adds
worker nodes when pending work needs resources that are not available, and removes nodes that stay idle.

In this Workspace there are 2 kinds of worker nodes: CPU nodes for chemistry and scoring, and L4 GPU nodes
for the models.

<img src="docs/images/ray/ray-cluster.svg" alt="Ray cluster" width="620">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

The current state of the cluster (run the cell again later to see nodes join and leave):

In [2]:
def cluster_table():
    rows = []
    for n in ray.nodes():
        if not n["Alive"]:
            continue
        r = n["Resources"]
        rows.append({"node": n["NodeManagerHostname"], "head": "node:__internal_head__" in r,
                     "CPUs": int(r.get("CPU", 0)), "GPUs": int(r.get("GPU", 0)),
                     "GPU type": next((k.split(":")[1] for k in r if k.startswith("accelerator_type:")), ""),
                     "memory (GB)": round(r.get("memory", 0) / 1e9)})
    return pd.DataFrame(rows).sort_values(["head", "GPUs"], ascending=False).reset_index(drop=True)

cluster_table()

,node,head,CPUs,GPUs,GPU type,memory (GB)
0,ip-10-0-108-36,True,0,0,,34
1,ip-10-0-163-17,False,8,1,L4,34
2,ip-10-0-131-2,False,8,1,L4,34


### 1.2 Tasks

Adding `@ray.remote` to a function turns it into a **task**. Calling `.remote()` schedules the task on any
node with a free CPU and returns an **`ObjectRef`** right away, a reference to a result that will exist
once the task finishes. `ray.get` waits for the result and returns it.

The example below uses a step that takes 1 second and runs it 16 times, first in plain Python and then as
16 Ray tasks. If no worker node is running yet, the first run also includes the time to start one; run the
cell again to see the time for the tasks alone.

In [3]:
def slow_step(i):
    time.sleep(1)          # stands in for 1 second of work, such as featurizing a molecule
    return i * i

t0 = time.time()
serial = [slow_step(i) for i in range(16)]
print(f"plain Python: {time.time() - t0:4.1f} s")

slow_task = ray.remote(slow_step)            # the same function as a Ray task
t0 = time.time()
refs = [slow_task.remote(i) for i in range(16)]
print(f"16 tasks submitted in {time.time() - t0:.3f} s, first reference: {refs[0]}")
parallel = ray.get(refs)
print(f"Ray tasks:    {time.time() - t0:4.1f} s, identical results: {parallel == serial}")

plain Python: 16.0 s
16 tasks submitted in 0.162 s, first reference: ObjectRef(fa31b89f94899135ffffffffffffffffffffffff0300000001000000)
Ray tasks:     3.7 s, identical results: True


A more realistic example: chemical descriptors for every compound in the 3 campaigns, computed with RDKit
in 16 chunks across the cluster. Each chunk records the node it ran on.

In [4]:
@ray.remote(num_cpus=1)
def describe(smiles: list[str]) -> pd.DataFrame:
    # Molecular weight, ring count and hydrogen-bond donors for each SMILES string
    import socket
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski
    rows = []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        rows.append({"smiles": s, "mol_weight": Descriptors.MolWt(m) if m else None,
                     "rings": Lipinski.RingCount(m) if m else None,
                     "hbond_donors": Lipinski.NumHDonors(m) if m else None, "node": socket.gethostname()})
    return pd.DataFrame(rows)

smiles = pd.concat([pd.read_csv(TARGETS[t].path("metadata")).smiles.drop_duplicates() for t in wh.TARGET_KEYS])
chunks = np.array_split(smiles.to_numpy(), 16)

t0 = time.time()
props = pd.concat(ray.get([describe.remote(list(c)) for c in chunks]))
print(f"{len(props)} molecules, 16 tasks, {time.time() - t0:.1f} s, {props.node.nunique()} node(s)")
props.drop(columns="node").describe().round(1)

1285 molecules, 16 tasks, 0.7 s, 2 node(s)


,mol_weight,rings,hbond_donors
count,1285.0,1285.0,1285.0
mean,375.7,3.6,1.1
std,122.5,1.7,0.7
min,73.1,0.0,0.0
25%,286.3,2.0,1.0
50%,352.2,3.0,1.0
75%,483.5,5.0,1.0
max,848.4,8.0,4.0


### 1.3 Resources and GPUs

Each task or actor declares the resources it needs: `num_cpus`, `num_gpus`, and optionally a GPU model with
`accelerator_type`. Ray places it on a node with those resources free. When no node has them, the
autoscaler starts one that does.

GPU requests can be fractional. With `num_gpus=0.5`, 2 processes share 1 GPU. Section 3 uses this for the
model stages: while 1 predictor prepares its next molecule on the CPU, the other uses the GPU.

The next cell requests 2 tasks with half an L4 each and continues without waiting for them. If no L4 node
is running, the autoscaler starts one now, which takes a few minutes. The results are collected at the end
of this section. Starting a GPU node now also means it is ready for Section 3.

In [5]:
@ray.remote(num_gpus=0.5)
def which_gpu(name):
    import socket, subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    return {"task": name, "node": socket.gethostname(), "GPU ids assigned by Ray": ray.get_gpu_ids(), "GPU": gpu}

gpu_refs = [which_gpu.options(accelerator_type=GPU).remote(n) for n in ("predictor A", "predictor B")]   # returns immediately
print(f"Requested 2 x 0.5 {GPU}. GPUs free right now: {ray.available_resources().get('GPU', 0)}")

Requested 2 x 0.5 L4. GPUs free right now: 2.0


### 1.4 Objects

Task results and arguments are stored as **objects** in the object store. A large input used by many
tasks, such as a metadata table, a set of fingerprints or model weights, can be stored once with
`ray.put`. Every task then receives the same `ObjectRef` and reads the shared copy on its node, so the
data is transferred once per node.

<img src="docs/images/ray/objects.png" alt="Objects in Ray" width="300">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

In [6]:
meta_all = pd.concat([pd.read_csv(TARGETS[t].path("metadata")).assign(target=t) for t in wh.TARGET_KEYS])
meta_ref = ray.put(meta_all)                     # stored once in the object store
print(f"metadata: {len(meta_all)} rows, stored as {meta_ref}")

@ray.remote
def count_phase(meta: pd.DataFrame, target: str) -> dict:
    # Ray passes the table itself to the task, resolved from the ObjectRef
    m = meta[meta.target == target]
    return {"target": TARGETS[target].name, "fragment-screen structures": int(m.fragment_screen.sum()),
            "follow-on structures": int((~m.fragment_screen.astype(bool)).sum())}

pd.DataFrame(ray.get([count_phase.remote(meta_ref, t) for t in wh.TARGET_KEYS]))

metadata: 2147 rows, stored as ObjectRef(00ffffffffffffffffffffffffffffffffffffff0300000001e1f505)


,target,fragment-screen structures,follow-on structures
0,EV-A71 2A protease,113,812
1,SARS-CoV-2 Mpro,59,569
2,MERS-CoV Mpro,0,594


### 1.5 Task dependencies and `ray.wait`

An `ObjectRef` can be passed directly into another task. Ray starts the second task when its input is
ready and moves the data to where it runs, so a whole graph of work can be submitted at once.
`ray.wait` returns results in the order they finish, which is useful for progress reporting or for acting
on early results.

In [7]:
@ray.remote
def heaviest(df: pd.DataFrame) -> pd.Series:
    return df.sort_values("mol_weight").iloc[-1]

# describe, then heaviest, submitted together; no ray.get in between
chained = [heaviest.remote(describe.remote(list(c))) for c in chunks[:6]]

pending, finished = chained, 0
while pending:
    ready, pending = ray.wait(pending, num_returns=1)
    finished += 1
    top = ray.get(ready[0])
    print(f"{finished}/6 finished: heaviest molecule {top.mol_weight:6.1f} Da  {top.smiles[:50]}")

1/6 finished: heaviest molecule  249.3 Da  Cc1ccc(OCC(=O)N2CCOCC2)c(C)c1
2/6 finished: heaviest molecule  412.9 Da  Cc1nn(C)c(C)c1CC(=O)N[C@@H](CNC(=O)CCl)c1ccc2ccccc
3/6 finished: heaviest molecule  415.4 Da  COC[C@]1(C)C[C@H](C(N)=O)CN1C(=O)Cc1cc(Br)ccc1SC
4/6 finished: heaviest molecule  393.9 Da  COc1cccc(Cl)c1CC(=O)N1C[C@H](C(N)=O)C[C@H](C(=O)NC
5/6 finished: heaviest molecule  385.3 Da  COc1cc(Br)c(CC(=O)N2CCC[C@H](C(N)=O)C2)cc1OC
6/6 finished: heaviest molecule  403.3 Da  CS(=O)(=O)c1ccc(Br)cc1CC(=O)N1CCC[C@H](C(N)=O)C1


### 1.6 Actors

An **actor** is a Python class with `@ray.remote`. Ray starts it once on 1 node, it keeps its state
between calls, and it processes method calls in order. Actors are the natural place for anything that is
expensive to load. A co-folding model with several GB of weights on a GPU is a typical example, and the
predictors in Section 3 run as actors.

The actor below loads the EV-A71 2A fragment screen once and then reports how similar any new compound is
to its closest fragment.

In [8]:
@ray.remote(num_cpus=1)
class FragmentMemory:
    def __init__(self, fragment_smiles):
        from rdkit import Chem, DataStructs
        from rdkit.Chem import rdFingerprintGenerator
        self.gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        self.DS, self.Chem = DataStructs, Chem
        self.fps = [self.gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in fragment_smiles]   # loaded once
        self.queries = 0

    def nearest(self, smiles):
        self.queries += 1                                                                      # kept between calls
        fp = self.gen.GetFingerprint(self.Chem.MolFromSmiles(smiles))
        return max(self.DS.BulkTanimotoSimilarity(fp, self.fps))

    def stats(self):
        return f"{len(self.fps)} fragments loaded, {self.queries} queries answered"

meta_ev71 = meta_all[meta_all.target == "ev71-2a"]
memory = FragmentMemory.remote(meta_ev71[meta_ev71.fragment_screen].smiles.unique().tolist())

late = meta_ev71[~meta_ev71.fragment_screen].drop_duplicates("compound_group").tail(5)
sims = ray.get([memory.nearest.remote(s) for s in late.smiles])
print(ray.get(memory.stats.remote()))
pd.DataFrame({"compound": late.compound_group, "similarity to closest fragment (0 to 1)": np.round(sims, 2)})

83 fragments loaded, 5 queries answered


,compound,similarity to closest fragment (0 to 1)
920,x7597a,0.40
921,x7608a,0.20
922,x7613a,0.27
923,x7622a,0.21
924,x7631a,0.25


Many follow-on compounds have a similarity well below 0.5 to every fragment. They contain chemistry the
fragment screen did not cover, which makes them the harder and more relevant test for a model. Results in
this notebook are therefore reported separately for fragments and follow-ons.

### 1.7 Fault tolerance

On a large cluster, failures are routine: a spot instance is reclaimed, a node runs out of memory, or an
external tool crashes on a single input. Ray retries failed tasks (`max_retries`, and `retry_exceptions=True` to
also retry Python exceptions) and can restart failed actors (`max_restarts`). In the cell below each attempt
fails 30% of the time, and all 20 tasks still complete. Ray logs each retry, so a few retry messages
appear above the result.

In [9]:
@ray.remote(max_retries=10, retry_exceptions=True)
def unreliable_step(i):
    import random
    if random.random() < 0.3:
        raise RuntimeError("simulated node failure")
    return i

print(f"{len(ray.get([unreliable_step.remote(i) for i in range(20)]))} of 20 tasks completed")

(raylet) Task unreliable_step failed. There are 10 retries remaining, so the task will be retried. Error: User exception:
ray::unreliable_step() (pid=7652, ip=10.0.163.17)
  File "/tmp/ipykernel_48443/3500811816.py", line 5, in unreliable_step
RuntimeError: simulated node failure
20 of 20 tasks completed


### 1.8 GPU placement and the Ray libraries

The 2 GPU tasks from 1.3 have had time to start. The cell waits up to 1 minute for them; if the GPU node is
still starting, run it again later.

In [10]:
ready, pending = ray.wait(gpu_refs, num_returns=len(gpu_refs), timeout=60)
if pending:
    print(f"The GPU node is still starting. Run this cell again in a few minutes. {cluster_table().GPUs.sum()} GPUs up.")
else:
    display(pd.DataFrame(ray.get(ready)))

,task,node,GPU ids assigned by Ray,GPU
0,predictor A,ip-10-0-163-17,[0],"NVIDIA L4, 23034 MiB"
1,predictor B,ip-10-0-163-17,[0],"NVIDIA L4, 23034 MiB"


Each task received half of an L4. Depending on what was free, Ray placed them on the same GPU or on 2
different GPUs. `ray.get_gpu_ids()` reports the device each process was given, and the Ray libraries use it
to put each model on the right GPU.

The rest of the workshop uses the 4 Ray libraries, each built on the primitives above:

| Library | Built on | Used for |
|---|---|---|
| **Ray Data** | Tasks for CPU stages, pools of actors for model stages, data streamed through the object store | Validate, predict with Boltz-2 and OpenBind-0, score against crystals |
| **Ray Train** | 1 actor per GPU, joined in a PyTorch distributed (DDP) group | Fine-tune OpenFold3 on 2 L4 GPUs |
| **Ray Tune** | 1 trial per configuration, each an actor or a complete Ray Train run, plus a scheduler that can stop trials early | Learning rate sweep and Boltz-2 inference settings |
| **Ray Serve** | Actor replicas behind an HTTP proxy, with replica autoscaling | Live Boltz-2 predictions and the 3D results viewer |

![Workflow](docs/images/workflow.png)

While the workshop runs, keep the **Metrics** page of the Ray Dashboard open in a separate browser tab
(the link is printed in Setup). These are the panels to watch, and what a good run looks like on each:

| Metrics panel | ✅ Looks good | ⚠️ Look into it |
|---|---|---|
| **Node GPU** (utilization) | Close to 100% on every GPU node while Boltz-2 or OpenBind-0 is predicting | Long gaps near 0%: the GPUs are waiting on CPU stages or on model loading |
| **Node GPU Memory (GRAM)** | Steady and below each L4's 24 GB | Climbing to the limit: lower `batch_size` or give each actor a full GPU |
| **Node Count** | GPU nodes join within a few minutes of the first GPU step | Stays flat: no GPUs are free, so set `RUN_LIVE = False` |
| **Scheduler Task State** / **Scheduler Actor State** | Tasks move from *pending* to *running* to *finished* | Many tasks stuck in *pending* for minutes: the cluster is short of the requested resources |
| **Object Store Memory** | Rises and falls as Ray Data streams batches through | Grows without dropping, or memory spills to disk |
| **Ray Data: Rows output / second** | Every operator keeps producing rows | One operator at 0 while the others wait: that operator is the bottleneck |

> 🎯 **The 3 most important ones:** GPU utilization (are the GPUs busy?), Node Count (did the GPU nodes start?)
> and Scheduler Task State (is any work stuck?).

## 2. The data: 3 drug discovery campaigns

A drug discovery campaign has 2 phases:

* **Fragment screen.** Hundreds of small molecules are soaked into protein crystals. Each hit is solved as
  a crystal structure during the experiment, so predicted poses add little at this stage.
* **Follow-on design.** Chemists grow the best fragments into larger molecules that bind more tightly,
  often hundreds of compounds over several months. Only some of them are crystallized, later and at
  significant cost. Reliable predicted poses are most valuable in this phase, and it is also where
  prediction is hardest.

All 3 targets are viral proteases, enzymes the virus needs to replicate. Each crystal structure serves as
ground truth. The models receive only the protein sequence and the compound's SMILES string.

In [11]:
rows, ligands = [], {}
for t in wh.TARGET_KEYS:
    meta, lig = load_ligands(TARGETS[t])
    ligands[t] = lig
    rows.append({"target": TARGETS[t].name,
                 "crystal structures": len(meta),
                 "unique compounds": len(lig),
                 **{st: int((lig.stage == st).sum()) for st in ["fragment", "early follow-on", "late follow-on"]},
                 "with measured affinity": int(lig.experimental_pKD.notna().sum())})
pd.DataFrame(rows).set_index("target")

,crystal structures,unique compounds,fragment,early follow-on,late follow-on,with measured affinity
target,,,,,,
EV-A71 2A protease,925,699,83,308,308,499
SARS-CoV-2 Mpro,628,308,57,145,106,164
MERS-CoV Mpro,594,285,0,128,157,142


Compound **x7259a** is a late EV-A71 2A follow-on with high measured affinity (pKD 7.9). Its crystal
structure is shown below: the dark sticks are the compound, and the thin lines are the protein residues
within 5 Å of it. Drag to rotate and scroll to zoom.

In [12]:
ev71 = wh.load_results("ev71-2a")         # full benchmark results, 1 row per crystal complex
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[0]
wh.show_complex(x7259a, models=())

The animation follows 1 compound series from a fragment to a potent follow-on, with each model's
prediction at every step.

![A fragment grown into a potent follow-on](docs/images/campaign.gif)

## 3. Predict and score with Ray Data

In this section you run the prediction pipeline on 8 late follow-on compounds from EV-A71 2A. The
predictions are written to a folder of your own, so both models run on the GPUs during the workshop.

A Ray Data `Dataset` is a table split into **blocks** that live in the object store. Each processing step
works on blocks in parallel across the cluster.

<img src="docs/images/ray/dataset-arch-with-blocks.svg" alt="Ray Data blocks" width="560">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

The pipeline has 4 steps, each added with `map_batches` or `flat_map`. Ray Data runs them as a stream:
blocks move to the next step as soon as they are ready, so CPU and GPU steps run at the same time.

<img src="docs/images/ray/streaming-topology.svg" alt="Streaming execution" width="560">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

| Step | Runs on | What it does |
|---|---|---|
| `validate` | CPU | Parses each SMILES string with RDKit |
| `Boltz2Predictor` | L4 GPU, 2 actors per GPU | Predicts pose and binding affinity with Boltz-2 |
| `OpenBind0` | L4 GPU, 2 actors per GPU | Predicts pose with OpenFold3 using the OpenBind-0 weights |
| `make_scorer` | CPU | Superposes each prediction on its crystal structure and computes the ligand RMSD |

The model steps use **actors** (`ActorPoolStrategy`), so each process loads its model once and reuses it
for every batch. `num_gpus=0.5` places 2 actors on each GPU. The full benchmark used the same classes on
8 GPUs:

![Pipeline architecture](docs/images/architecture.png)

In [13]:
# x7259a plus 7 other late follow-ons with a curated crystal structure (non-covalent)
lig = ligands["ev71-2a"]
curated = set(ev71.complex_name)
pool = lig[(lig.stage == "late follow-on") & lig.curated_any & ~lig.covalent.astype(bool) & (lig.ligand_id != "x7259a")]
picked = pd.concat([lig[lig.ligand_id == "x7259a"], pool.sample(7, random_state=1)]).reset_index(drop=True)
picked["complexes"] = picked.complexes.map(lambda cs: [c for c in cs if c in curated])
picked[["ligand_id", "smiles", "stage", "experimental_pKD"]]

,ligand_id,smiles,stage,experimental_pKD
0,x7259a,CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1,late follow-on,7.9380
1,x7025a,CSc1ncnc(N2CC[C@@H](OC(F)F)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1,late follow-on,6.0481
2,x6934a,CSc1ccccc1[C@@H](NC(=O)C1CC1)C(=O)N1CC[C@H](C(N)=O)C1,late follow-on,4.6120
3,x7096a,CSc1ncnc(N2CCC(C(=O)N(C(C)C)C(C)C)CC2)c1CC(=O)N1CCC[C@H]...,late follow-on,6.2253
4,x6827a,CSc1ccc(C2=CC[NH2+]CC2)cc1CC(=O)N1CCC[C@H](C(N)=O)C1,late follow-on,6.3598
5,x7321a,CC[C@H]1CS[C@H](C(N)=O)CN1C(=O)Oc1cc(Br)ccc1SC,late follow-on,4.1702
6,x7154a,CSc1ncnc(N2CCC(N3CCOC3=O)CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1,late follow-on,5.9235
7,x7379a,CSc1ccc(-c2cn[nH]c2)cc1OC(=O)N1C[C@@H](C(N)=O)C[C@H]2C[C...,late follow-on,5.6262


In [14]:
MY_PRED = wh.MINE / "predictions"

class OpenBind0(OF3Predictor):
    # OpenFold3 with the OpenBind-0 weights, writing to your own folder
    def __init__(self, out_dir):
        super().__init__(key="openbind0", ckpt_path=str(OF3_OB_CKPT), target="ev71-2a")
        self.out = Path(out_dir)
        self.out.mkdir(parents=True, exist_ok=True)

gpu_step = dict(batch_size=2, batch_format="pandas", num_gpus=0.5, num_cpus=3, accelerator_type=GPU,
                compute=ray.data.ActorPoolStrategy(size=2, max_tasks_in_flight_per_actor=1))

ds = (ray.data.from_pandas(picked[["ligand_id", "smiles", "complexes", "stage", "experimental_pKD"]])
      .repartition(4)
      .map_batches(validate, batch_format="pandas")                       # CPU
      .map_batches(Boltz2Predictor, **gpu_step,                            # GPU
                   fn_constructor_kwargs={"target": "ev71-2a", "out_dir": str(MY_PRED / "boltz2")})
      .map_batches(OpenBind0, **gpu_step,                                  # GPU
                   fn_constructor_kwargs={"out_dir": str(MY_PRED / "openbind0")})
      .flat_map(make_scorer(["boltz2", "openbind0"], "ev71-2a")))          # CPU
ds    # execution is lazy: nothing runs until results are requested

FlatMap(score)
+- MapBatches(OpenBind0)
   +- MapBatches(Boltz2Predictor)
      +- MapBatches(validate)
         +- Repartition
            +- Dataset(
                  num_rows=8,
                  schema={
                     ligand_id: object,
                     smiles: object,
                     complexes: object,
                     stage: object,
                     experimental_pKD: float64
                  }
               )

⏱ **The next cell takes about 5 to 10 minutes on the first run**, mostly for GPU nodes to start and models
to load. On the **Metrics** page, *Node Count* shows the GPU nodes joining, *Node GPU* shows them start predicting,
and *Ray Data: Rows output / second* shows progress for each pipeline step.

If GPU nodes are not available, set `RUN_LIVE = False`. The cell then loads the stored predictions for
the same 8 compounds from the full benchmark, and the rest of the notebook works the same way.

In [15]:
RUN_LIVE = True

t0 = time.time()
mine = ds.to_pandas() if RUN_LIVE else ev71[ev71.ligand_id.isin(picked.ligand_id)].reset_index(drop=True)
print(f"{mine.ligand_id.nunique()} compounds, {len(mine)} crystal complexes scored in {(time.time() - t0) / 60:.1f} min")
for k in ("boltz2", "openbind0"):
    print(f"  {wh.MODEL_NAMES[k]:12s} {mine.drop_duplicates('ligand_id')[f'{k}_status'].value_counts().to_dict()}")

view = mine[["complex_name", "boltz2_lig_rmsd", "openbind0_lig_rmsd", "boltz2_affinity_pred_value"]].copy()
view.columns = ["complex", "Boltz-2 RMSD (Å)", "OpenBind-0 RMSD (Å)", "Boltz-2 affinity (log10 IC50, µM)"]
(view.style.format(precision=2)
     .map(lambda v: "color:#2E7D4F;font-weight:600" if v < 2 else "color:#B3412E",
          subset=["Boltz-2 RMSD (Å)", "OpenBind-0 RMSD (Å)"]))

2026-10-16 15:15:32,027	WARNING map_operator.py:958 -- Specifying both num_cpus and num_gpus for map tasks is experimental, and may result in scheduling or stability issues. Please report any issues to the Ray team: https://github.com/ray-project/ray/issues/new/choose
2026-10-16 15:15:32,028	WARNING map_operator.py:958 -- Specifying both num_cpus and num_gpus for map tasks is experimental, and may result in scheduling or stability issues. Please report any issues to the Ray team: https://github.com/ray-project/ray/issues/new/choose
2026-10-16 15:15:32,030	INFO streaming_executor.py:214 -- Starting execution of Dataset dataset_034h91B4wAucMde5eJk7rE_0. Full logs are in /tmp/ray/session_2026-10-16_15-00-44_454583_31303/logs/ray-data
2026-10-16 15:15:32,031	INFO streaming_executor.py:215 -- Execution plan of Dataset dataset_034h91B4wAucMde5eJk7rE_0: InputDataBuffer[Input] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[MapBatches(validate)] -> ActorPoolMapOperator[MapBatches(Boltz

{"asctime":"2026-10-16 15:15:32,058","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(Boltz2Predictor))' and ID: '5b469872b423e319b7089b7503000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.","filename":"core_worker.cc","lineno":2297}
2026-10-16 15:15:32,082	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.
2026-10-16 15:15:32,291	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_034h91B4wAucMde5eJk7rE_0 =======
2026-10-16 15:15:32,292	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-10-16 15:15:32,292	INFO logging_progress.py:227 -- Active & requested resources: 0/0 CPU, 0.0B/0.0B object store (pending: 12 CPU, 2 GPU)
2026-10-16 15:15:32,293	INFO logging_progress.py:181 -- 
2026-10-16 15:15:32,294	INFO logging_

**Ligand RMSD** is the average distance between each predicted atom and the same atom in the crystal
structure, after the predicted protein is superposed on the crystal protein. A pose within **2 Å** (green)
is counted as correct. For scale, a carbon-carbon bond is about 1.5 Å long.

How many predictors to place on each GPU depends on the model. On 1 L4, 2 OpenFold3 processes nearly double
throughput (181 to 327 ligands per hour) because the GPU spends less time idle between molecules. Boltz-2
already keeps the GPU 75% busy with 1 process, and a second process makes no difference (88 and 83 per
hour). The pipeline settings come from measurements like these:

![Throughput](docs/images/throughput.png)

GPU utilization per node during the full MERS-CoV Mpro benchmark. Nodes were added as work arrived and
released as each model finished:

![GPU cluster](docs/images/gpu_cluster.png)

In [ ]:
# Your predictions superposed on the crystal: dark = crystal, blue = Boltz-2, orange = OpenBind-0
wh.show_complex(mine[mine.complex_name == "A71EV2A-x7259a"].iloc[0])

For x7259a, OpenBind-0 reproduces the crystal pose within a fraction of an Ångström, while Boltz-2 places
the compound in a different part of the protein. The next cell shows 2 more compounds from your run.

In [ ]:
for _, row in mine.drop_duplicates("ligand_id").iloc[1:3].iterrows():
    wh.show_complex(row, height=340)

## 4. Results across 2,147 crystal structures

The same pipeline was run on all 3 campaigns before the workshop, taking 1 to 2 hours per target on 8 L4
GPUs. The results below use the curated set: complexes with a valid crystal reference and no suspected
crystallographic artefact. A third model is included, an OpenFold3 checkpoint released by the OpenBind
team after fine-tuning on the EV-A71 2A fragment screen. Section 5 shows how such a checkpoint is trained.

In [ ]:
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
table = wh.glance(results)
print("Share of poses within 2 Å of the crystal structure (%)")
table.style.format("{:.1f}").background_gradient(cmap="Greens", vmin=0, vmax=100, axis=None)

Key observations:

* Performance depends strongly on the target. Boltz-2 is accurate on both coronavirus proteases and
  predicts only 3% of EV-A71 2A follow-on poses correctly.
* The fine-tuned model reaches 79% on EV-A71 2A follow-ons, compared with 3% for Boltz-2. On the 2 Mpro
  targets it reaches 62% to 69%, below the 83% to 89% of the general models.
* The fine-tuned model's 81% on EV-A71 2A fragments is measured on its own training data. All other values
  are on compounds the model has not seen.

For a new project, this suggests benchmarking candidate models on the project's own fragment-screen
structures before relying on them for follow-on design. This pipeline can run that benchmark on new
crystal data.

Results by campaign stage for EV-A71 2A:

In [ ]:
wh.by_stage(results["ev71-2a"]).style.format("{:.1f}").set_caption("EV-A71 2A: share of poses within 2 Å by stage (%)")

Cumulative accuracy and accuracy by stage for the 2 targets with a fragment screen:

![Pose accuracy](docs/images/pose_accuracy.png)

**Binding affinity.** Boltz-2 also predicts binding affinity. The Spearman correlation below measures how
well the predicted values rank compounds compared with the measured values: 1 is an identical ranking and
0 is no relationship.

In [ ]:
rho = {}
for t, df in results.items():
    a = df.drop_duplicates("ligand_id")[["experimental_pKD", "boltz2_affinity_pred_value"]].astype(float).dropna()
    # Boltz-2 reports log10(IC50 in µM), where lower means tighter binding, so the sign is flipped
    rho[TARGETS[t].name] = {"Spearman": round((-a.boltz2_affinity_pred_value).corr(a.experimental_pKD, "spearman"), 2),
                            "compounds": len(a), "measurement": TARGETS[t].affinity_label}
pd.DataFrame(rho).T

![Affinity](docs/images/affinity.png)

The next cell finds other EV-A71 2A follow-on complexes where the models disagree most and shows the
compound with the highest measured affinity among them.

In [ ]:
ev = results["ev71-2a"]
disagree = ev[(ev.openbind0_lig_rmsd < 1) & (ev.boltz2_lig_rmsd > 8) & ~ev.fragment_screen]
disagree = disagree[disagree.ligand_id != "x7259a"]      # x7259a was shown above
print(f"{len(disagree)} other EV-A71 2A follow-on complexes with OpenBind-0 under 1 Å and Boltz-2 over 8 Å")
wh.show_complex(disagree.sort_values("experimental_pKD", ascending=False).iloc[0],
                models=("boltz2", "openbind0", "of3ft"))

In [ ]:
# Enter any complex name from the 3 targets, for example "MERS-a1994b" or "SARS-5rf4-a"
CID = "MERS-a1994b"
row = pd.concat(results.values()).query("complex_name == @CID").iloc[0]
wh.show_complex(row, models=("boltz2", "openbind0", "of3ft"))

## 5. Fine-tune OpenFold3 with Ray Train

Every campaign begins with a fragment screen that produces dozens of crystal structures of the target
protein. Fine-tuning uses these structures to adapt a general model to that protein before the follow-on
phase.

This run starts from the OpenBind-0 weights and trains on the **66 EV-A71 2A fragment-screen complexes**.
OpenFold3's PyTorch Lightning training code runs unchanged inside Ray Train, which starts 1 worker per GPU,
sets up the distributed process group and collects metrics and checkpoints.

<img src="docs/images/ray/train-overview.png" alt="Ray Train" width="560">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

The core of `finetune_ray_train.py`:

```python
trainer = TorchTrainer(
    train_loop_per_worker,                    # OpenFold3's Lightning runner with RayDDPStrategy
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True, accelerator_type="L4"),
    run_config=RunConfig(storage_path=..., checkpoint_config=CheckpointConfig(num_to_keep=1)))
result = trainer.fit()
```

Each worker calls `ray.train.report(metrics)` after every optimizer step with the loss and the
protein-ligand lDDT, and the final weights are returned as a Ray `Checkpoint`.

⏱ **The next cell runs a 20-step fine-tune on 2 L4 GPUs, about 5 to 8 minutes including startup.**
The full run is 150 steps and takes about 30 minutes. Set `RUN_FINETUNE = False` to skip the run and use
the recorded 150-step run below.

In [ ]:
RUN_FINETUNE = True
import json

recorded = sorted((SHARED / "finetune" / "runs").glob("ev71-l4x2-150/checkpoint_*/loss_curve.json"))
recorded = pd.DataFrame(json.loads(recorded[-1].read_text())) if recorded else None

if RUN_FINETUNE:
    import finetune_ray_train as ft
    args = ft.parser().parse_args(["--num-workers", "2", "--max-steps", "20",
                                   "--name", f"workshop-{time.strftime('%H%M%S')}",
                                   "--storage", str(wh.MINE / "finetune"), "--accelerator", GPU])
    ft.check_inputs(args)
    t0 = time.time()
    result = ft.make_trainer(args).fit()
    print(f"\n{args.max_steps} steps on {args.num_workers} GPUs in {(time.time() - t0) / 60:.1f} min")
    print(f"Weights: {result.checkpoint.path}/of3_ev71_ft.pt")
    yours = result.metrics_dataframe

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, col, name in zip(axes, ["loss", "lddt_inter_protein_ligand_step"],
                         ["Training loss (lower is better)", "Protein-ligand lDDT (higher is better)"]):
    if recorded is not None:
        r = recorded[recorded.step > 1]
        ax.plot(r.step, r[col], color="#1F5A96", alpha=0.25, lw=0.8)
        ax.plot(r.step, r[col].rolling(20, min_periods=5).mean(), color="#1F5A96", lw=2,
                label="recorded run, 150 steps")
    if RUN_FINETUNE:
        y = yours.dropna(subset=[col])
        y = y[y.step > 1]
        ax.plot(y.step, y[col], "o-", color="#B3412E", ms=3, lw=1.2, label="your run, 20 steps")
    ax.set_title(name, loc="left", fontsize=11)
    ax.set_xlabel("optimizer step")
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False)
plt.tight_layout()

Because training starts from OpenBind-0, which already performs well on this protein, the training loss
changes little. Model quality is measured on held-out compounds. The released fine-tuned checkpoint in
Section 4 was trained on fragments only and predicts 79% of EV-A71 2A follow-on poses correctly, compared
with 3% for Boltz-2. On the 2 proteases it was not trained on, it reaches 62% to 69%. Fine-tuned models
are best used on the target they were trained for.

To benchmark your own weights on every EV-A71 2A compound, submit an Anyscale Job:

```bash
anyscale job submit -f job.yaml --name my-of3ray -- \
  python aiddd_pipeline.py --models of3ray,openbind0 --of3ray-ckpt <your weights>
```

## 6. Hyperparameter and inference sweeps with Ray Tune

Ray Tune runs many trials in parallel, each with its own resources, and its schedulers stop trials that
fall behind. Both sweeps below were run before the workshop with `tune_finetune.py` (35 minutes on 4 L4
GPUs) and `tune_boltz.py` (62 minutes on 8 L4 GPUs).

<img src="docs/images/ray/tune_flow.png" alt="Ray Tune" width="620">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

**Sweep 1: fine-tuning learning rate.** If the learning rate is too low the model barely changes, and if
it is too high the model loses what it learned in pretraining. Each trial is a complete Ray Train run. The
ASHA scheduler stops trials whose lDDT falls behind, and the remaining trial is evaluated on 32 held-out
early follow-on complexes.

In [ ]:
lr = pd.read_csv(SHARED / "finetune" / "runs" / "tune-of3-ev71" / "summary.csv")
lr["result"] = np.where(lr.val_success.notna(),
                        (100 * lr.val_success).round(1).astype(str) + "% within 2 Å on held-out compounds",
                        "stopped by ASHA at step " + lr.steps.astype(int).astype(str))
lr[["lr", "steps", "lddt_avg20", "result"]].style.format({"lr": "{:.0e}", "lddt_avg20": "{:.3f}"}).hide()

ASHA stopped 3 of the 4 trials early, so the sweep used about 450 GPU-steps compared with 600 for a full
grid. The selected learning rate, 3e-4, reaches 93.8% on held-out compounds, the same as OpenBind-0 and
higher than the released fine-tune (87.5%).

**Sweep 2: Boltz-2 inference settings on EV-A71 2A.** Each trial is 1 combination of recycling steps,
diffusion samples and physics-based steering. All 8 trials ran at the same time on 8 L4 GPUs.

In [ ]:
bz = pd.read_csv(SHARED / "tune" / "runs" / "tune-boltz2-ev71" / "summary.csv")
bz["% within 2 Å"] = (100 * bz.success).round(1)
bz["GPU minutes"] = bz.minutes.round(0)
(bz[["setting", "% within 2 Å", "median_rmsd", "GPU minutes"]].rename(columns={"median_rmsd": "median RMSD (Å)"})
   .style.format({"% within 2 Å": "{:.1f}", "median RMSD (Å)": "{:.1f}", "GPU minutes": "{:.0f}"}).bar(subset=["% within 2 Å"], color="#9fb6cf", vmin=0, vmax=100).hide())

The best setting raises Boltz-2 from 0% to 13.5% on this sample, at about 6 times the GPU time of the
default. OpenBind-0 reaches 92.7% with default settings. The median error stays near 4 Å for every
setting: Boltz-2 places the compound near the pocket in the wrong orientation, and additional sampling
rarely corrects it. On this target, model choice or fine-tuning has a much larger effect than inference
compute.

To run either sweep as an Anyscale Job:

```bash
anyscale job submit -f job.yaml --name tune-ft    -- python tune_finetune.py
anyscale job submit -f job.yaml --name tune-boltz -- python tune_boltz.py
```

## 7. Live prediction service and 3D viewer with Ray Serve

The final step makes the models available to chemists through a web application. `serve_viewer.py` is a
Ray Serve application with 2 deployments:

| Deployment | Resources | Function |
|---|---|---|
| `Viewer` | 1 CPU | Web page with a table of every crystal complex, a 3D view of each prediction on its crystal structure, and a SMILES input |
| `Boltz2Live` | 1 L4 per replica, autoscaling from 1 to 4 replicas | Boltz-2 pose and affinity for a new molecule, using the same `Boltz2Predictor` class as the pipeline |

`Viewer` calls `Boltz2Live` through a deployment handle, so the web page and the GPU model scale
separately. Repeated molecules are returned from a cache.

<img src="docs/images/ray/serve-architecture.svg" alt="Ray Serve architecture" width="600">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

⏱ **Deployment takes 1 to 3 minutes** if a GPU node needs to start.

In [ ]:
from ray import serve
import serve_viewer

# The 2 deployments, composed: the Viewer receives a handle to the Boltz-2 deployment
live = serve_viewer.Boltz2Live.options(ray_actor_options={"num_gpus": 1, "accelerator_type": GPU})
serve.run(serve_viewer.Viewer.bind(live.bind()), name="aiddd", route_prefix="/", blocking=False)
viewer_url = wh.serve_link()

In the viewer, select a model tab and click a complex to compare the predicted compound (green) with the
crystal structure (orange). The viewer runs on the same Ray cluster as this notebook.

**Prediction from Python.** The next cell sends a new molecule to the service: x7259a with its
methylthio group (S-CH3) replaced by a methoxy group (O-CH3). This compound has no crystal structure. The
cell shows the predicted pose next to the crystal pose of x7259a, along with Boltz-2's predicted affinity.

In [ ]:
import requests

def predict(smiles):
    t0 = time.time()
    r = requests.post("http://localhost:8000/predict", json={"smiles": smiles}, timeout=900)
    out = r.json()
    if r.status_code != 200:
        raise RuntimeError(out.get("error"))
    m = out["metrics"]
    print(f"{out['status']} in {time.time() - t0:.0f} s · ligand ipTM {m.get('ligand_iptm', float('nan')):.2f} · "
          f"affinity {m.get('affinity_pred_value', float('nan')):.2f} log10 IC50 (µM) · "
          f"P(binder) {m.get('affinity_probability_binary', float('nan')):.2f}")
    return out

NEW = "COc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7259a with S replaced by O
out = predict(NEW)
wh.show_analog(SHARED / "predictions" / "live" / f"{out['ligand_id']}.cif", x7259a)

The grey ribbon and dark sticks are the crystal structure of x7259a. The green sticks are the Boltz-2
prediction for the new molecule, superposed on that crystal. Section 4 showed that Boltz-2 rarely predicts
EV-A71 2A follow-on poses correctly, so for this target a prediction from OpenBind-0 or a fine-tuned model
would be the better choice.

**Try your own molecule.** Replace the SMILES string below (up to 300 characters). A new molecule takes
about 1 minute, and a repeated one returns immediately from the cache.

In [ ]:
MY_SMILES = "CNC(=O)c1nn(C)c2ccccc12"
out = predict(MY_SMILES)
wh.show_prediction(out["cif"], label=MY_SMILES)

**Sharing the viewer with the room.** The instructor deploys the same application as an Anyscale Service,
which provides 1 public URL with autoscaling GPU replicas:

```bash
anyscale service deploy -f service.yaml          # prints the URL to share
anyscale service terminate -n aiddd-viewer       # after the session
```

The application code is identical in both cases: `serve.run` in a Workspace during development, and an
Anyscale Service in production.

## 8. Summary and next steps

In this workshop you used 1 Ray cluster to:

1. Run a streaming CPU and GPU pipeline with **Ray Data**, with 2 model actors per GPU. At full scale the
   pipeline processed 2,147 crystal structures on 8 L4 GPUs.
2. Benchmark 3 models on 3 targets and found that accuracy on follow-on compounds varies widely by target
   and model.
3. Fine-tune OpenFold3 on a fragment screen with **Ray Train** on 2 GPUs.
4. Select a learning rate with **Ray Tune** and early stopping, and measure that 6 times more inference
   compute raised Boltz-2 from 0% to 13.5% on EV-A71 2A.
5. Deploy Boltz-2 and a 3D viewer as an autoscaling service with **Ray Serve**.

The [README](README.md) is the complete runbook for this repository. It covers running every step as an
Anyscale Job, adding a new target from your own crystal data, and interpreting each output.

Run the last cell to stop the service and release the GPU nodes:

In [ ]:
serve.shutdown()
print("Serve application stopped. Idle GPU nodes will be released within a few minutes.")